In [23]:
import pandas as pd
import numpy as np
from scipy import stats
from typing import Optional, Literal

Load original pick data CSV into dataframe.

In [ ]:
# Adjust the path name as needed to pull in the original CSV file.
filename = r"C:\Users\LindseyBuss\Documents\OBETA_Project\003 pick_data.csv"
column_names = ['product_id', 'warehouse_section', 'origin', 'order_number', 'position_in_order', 'pick_volume', 'quantity_unit', 'date']
data_types = {
    'product_id': 'string', 
   'warehouse_section': 'category',
   'origin': 'category',
   'order_number': 'string',
   'position_in_order':'int64',
   'pick_volume': 'int64',
   'quantity_unit': 'string',
   'date': 'string'
}
original_pick_data_df = pd.read_csv(filename, names=column_names, header=None, dtype=data_types, parse_dates=['date'])
print(original_pick_data_df.head())

  product_id warehouse_section origin order_number  position_in_order  \
0     000002               SHL     48     07055448                  1   
1     000002               SHL     48     07055448                  1   
2     000002               SHL     48     07055448                  1   
3     000002               SHL     48     07055448                  1   
4     000002               SHL     48     07055448                  1   

   pick_volume quantity_unit                date  
0           29            St 2017-06-30 11:15:24  
1           30            St 2017-06-30 11:22:35  
2           30            St 2017-06-30 12:04:50  
3           20            St 2017-06-30 12:04:51  
4           30            St 2017-06-30 12:05:02  


Add unique pick IDs, and generate unique order numbers.

In [25]:
# Add unique pick IDs.
original_pick_data_df['pick_id'] = range(1, len(original_pick_data_df) + 1)

# Extract year of order to create new unique order IDs.
original_pick_data_df['year_of_order'] = original_pick_data_df['date'].dt.year.astype(str)

original_pick_data_df['updated_order_number'] = original_pick_data_df[['order_number', 'year_of_order']].astype(str).agg('-'.join, axis=1)
print(original_pick_data_df.head())

  product_id warehouse_section origin order_number  position_in_order  \
0     000002               SHL     48     07055448                  1   
1     000002               SHL     48     07055448                  1   
2     000002               SHL     48     07055448                  1   
3     000002               SHL     48     07055448                  1   
4     000002               SHL     48     07055448                  1   

   pick_volume quantity_unit                date  pick_id year_of_order  \
0           29            St 2017-06-30 11:15:24        1          2017   
1           30            St 2017-06-30 11:22:35        2          2017   
2           30            St 2017-06-30 12:04:50        3          2017   
3           20            St 2017-06-30 12:04:51        4          2017   
4           30            St 2017-06-30 12:05:02        5          2017   

  updated_order_number  
0        07055448-2017  
1        07055448-2017  
2        07055448-2017  
3        0

Drop null values, duplicates, original order number and picks with a volume of zero.


In [ ]:
original_pick_data_df = original_pick_data_df.drop(columns='order_number')
original_pick_data_df = original_pick_data_df.dropna()
original_pick_data_df = original_pick_data_df.drop_duplicates()
original_pick_data_df = original_pick_data_df[original_pick_data_df['pick_volume'] != 0]

cleaned_pick_data = original_pick_data_df
# Export the cleaned pick data without outliers if desired.
# cleaned_pick_data.to_parquet()

Flag the pick volume outliers (grouped by quantity unit).

In [27]:

def flag_outliers_by_unit(
    df: pd.DataFrame,
    value_col: str,
    *,
    group_col: str = "quantity_unit",
    method: Literal["zscore", "modified"] = "zscore",
    threshold: Optional[float] = None,
    ddof: int = 0,
    min_group_size: int = 3,
    separate_sides: bool = False,
    flag_col: Optional[str] = None,
) -> pd.DataFrame:
    """
    Flag outliers in `value_col` *per group* defined by `group_col` (e.g., quantity_unit).

    Parameters
    ----------
    df : pd.DataFrame
        Input DataFrame.
    value_col : str
        Numeric column to analyze for outliers.
    group_col : str
        Grouping column (default: 'quantity_unit').
    method : {'zscore', 'modified'}
        'zscore' = mean/std; 'modified' = median/MAD (robust).
    threshold : float, optional
        Threshold for the chosen method. Defaults: 3.0 for zscore, 3.5 for modified.
    ddof : int
        ddof for std in zscore method.
    min_group_size : int
        Minimum group size to compute outliers. Smaller groups are marked as non-outliers.
    separate_sides : bool
        If True, adds two columns: lower/upper outliers (instead of a single boolean).
    flag_col : str, optional
        Name of the output flag column when separate_sides=False.
        Defaults to f'{value_col}_is_outlier_by_{group_col}'.

    Returns
    -------
    pd.DataFrame
        Copy of df with added outlier flag column(s), computed per group.

    Notes
    -----
    - NaNs in value_col are ignored in stats; rows with NaN are flagged as non-outliers.
    - Constant groups (std=0 or MAD=0) produce no outliers.
    """

    if threshold is None:
        threshold = 3.0 if method == "zscore" else 3.5
    if flag_col is None and not separate_sides:
        flag_col = f"{value_col}_is_outlier_by_{group_col}"

    out = df.copy()

    # Ensure numeric dtype (will convert errors to NaN)
    x = pd.to_numeric(out[value_col], errors="coerce")

    def compute_group_flags(g: pd.Series) -> pd.DataFrame:
        # g is the value_col Series for one group
        # Build a DataFrame with aligned index for returning multiple columns
        res = pd.DataFrame(index=g.index)

        # Respect minimum group size
        valid = g.dropna()
        if valid.size < min_group_size:
            if separate_sides:
                res["is_lower_outlier"] = False
                res["is_upper_outlier"] = False
            else:
                res[flag_col] = False
            return res

        if method == "zscore":
            mean = valid.mean()
            std = valid.std(ddof=ddof)
            if std == 0 or np.isnan(std):
                z = pd.Series(0.0, index=g.index)
            else:
                z = (g - mean) / std
        elif method == "modified":
            med = valid.median()
            mad = (valid - med).abs().median()
            if mad == 0 or np.isnan(mad):
                z = pd.Series(0.0, index=g.index)
            else:
                z = 0.6745 * (g - med) / mad
        else:
            raise ValueError("method must be 'zscore' or 'modified'")

        # Build masks; treat NaNs as non-outliers
        if separate_sides:
            res["is_lower_outlier"] = z.lt(-threshold).fillna(False)
            res["is_upper_outlier"] = z.gt(threshold).fillna(False)
        else:
            res[flag_col] = z.abs().gt(threshold).fillna(False)

        return res

    flags = x.groupby(out[group_col], dropna=False).apply(compute_group_flags)
    # groupby.apply returns a nested index; drop the group level
    flags.index = flags.index.get_level_values(-1)

    # Merge flags back
    for col in flags.columns:
        out[col] = flags[col]

    return out


  


Flag the outliers in the pick data.

In [28]:

pick_data_with_outliers = flag_outliers_by_unit(
    original_pick_data_df,
    value_col="pick_volume",
    group_col="quantity_unit",
    method="zscore",
    threshold=3.0,
)
pick_data_with_outliers.head()


,product_id,warehouse_section,origin,position_in_order,pick_volume,quantity_unit,date,pick_id,year_of_order,updated_order_number,pick_volume_is_outlier_by_quantity_unit
0,000002,SHL,48,1,29,St,2017-06-30 11:15:24,1,2017,07055448-2017,False
1,000002,SHL,48,1,30,St,2017-06-30 11:22:35,2,2017,07055448-2017,False
2,000002,SHL,48,1,30,St,2017-06-30 12:04:50,3,2017,07055448-2017,False
3,000002,SHL,48,1,20,St,2017-06-30 12:04:51,4,2017,07055448-2017,False
4,000002,SHL,48,1,30,St,2017-06-30 12:05:02,5,2017,07055448-2017,False


Export the cleaned pick data with pick volume outliers flagged.

In [ ]:
cleaned_pick_data = pick_data_with_outliers
# cleaned_pick_data.to_csv('cleaned_pick_data.csv')

Subset data and prepare order summary information. Subset by year to ensure all picks from the same order are processed in the same batch.

In [ ]:
# Create a list of the years represented in the pick data set.
years = cleaned_pick_data['year_of_order'].unique()


Define function to generate summary information about each order to reveal complexity and fulfillment time.

In [32]:
def summarize_year(df):
    order_summary_data = df.groupby('updated_order_number').agg(
        origin = ('origin', 'first'),
        num_picks = ('pick_volume', 'sum'),
        num_products = ('product_id', 'nunique'),
        num_sections = ('warehouse_section', 'nunique'),
        num_positions = ('position_in_order', 'nunique'),
        time_of_first_pick = ('date', 'min'),
        time_of_last_pick = ('date', 'max'))

    order_summary_data['time_to_fulfil'] = order_summary_data['time_of_last_pick'] - order_summary_data['time_of_first_pick']
    order_summary_data['date'] = order_summary_data['time_of_first_pick'].dt.date

    return order_summary_data

Summarize the orders within each year and combine into overall order summary dataframe.

In [ ]:
order_summaries_by_year = []

for year in years:
    df_subset = cleaned_pick_data[cleaned_pick_data['year_of_order'] == year]
    summarized_year = summarize_year(df_subset)
    order_summaries_by_year.append(summarized_year)

final_order_data = pd.concat(order_summaries_by_year)
print(final_order_data.head())
# Export final order summary data.
# final_order_data.to_parquet()



                     origin  num_picks  num_products  num_sections  \
updated_order_number                                                 
04154598-2017            48          1             1             1   
04155767-2017            48          2             1             1   
04155790-2017            48        282             7             2   
04155811-2017            48         14            10             2   
04155813-2017            48         69             5             2   

                      num_positions  time_of_first_pick   time_of_last_pick  \
updated_order_number                                                          
04154598-2017                     1 2017-01-02 14:22:45 2017-01-02 14:22:45   
04155767-2017                     1 2017-01-05 19:03:09 2017-01-05 19:03:09   
04155790-2017                     7 2017-01-02 12:20:57 2017-01-02 12:25:23   
04155811-2017                    10 2017-01-02 12:19:34 2017-01-02 12:34:51   
04155813-2017                     5

Import product info, clean, and create product_group table.

In [ ]:
# Load product data into df. Use Latin 1 encoding to avoid issue with reading umlauts. 
# Update the file path as needed.
filename = r"C:\Users\LindseyBuss\Documents\OBETA_Project\002 product_data.csv"
column_names = ['product_id', 'product_description', 'product_group']
data_types = {
    'product_id': 'string', 
   'product_description': 'string',
   'product_group': 'category'
}
product_groups_df = pd.read_csv(filename, names=column_names, header=None, dtype=data_types, encoding='latin-1')

print(product_groups_df.head())

  product_id             product_description               product_group
0     000052        PUNCH II MSW1 1500mm PUN                 35_Leuchten
1    1036628              H05VV-F2X1 5WS 50M           16_Sonderverkäufe
2    1052053                 H07RN-F3G1 100M           16_Sonderverkäufe
3     110109  SIEM DELTA Doppelta 2S 5TD2111  32_Schalter_Steckvorrichtg
4     110125  SIEM PLUS Wip Univ ews 5TG7581  32_Schalter_Steckvorrichtg


In [67]:
def clean_products(df):
    # Remove missing values
    df = df.dropna()

    # Remove duplicates
    df = df.drop_duplicates()

    # Drop product descriptions
    df = df.drop('product_description', axis=1)

    # Truncate product group, leaving only the group number
    df['product_group'] = df['product_group'].apply(lambda x: x.split('_')[0])

    return df

In [68]:
cleaned_product_df = clean_products(product_groups_df)
print(cleaned_product_df.head(10))

  product_id product_group
0     000052            35
1    1036628            16
2    1052053            16
3     110109            32
4     110125            32
5     110128            32
6     110129            32
7     110136            32
8     110146            32
9     110150            32


Generate product data table.

In [ ]:
product_data = pick_data_with_outliers[['product_id', 'warehouse_section', 'quantity_unit']]
product_data['product_group_num'] = product_data['product_id'].map(cleaned_product_df.set_index('product_id')['product_group'])
product_data = product_data.drop_duplicates()
print(product_data.head())

# Export product_data


C:\Users\LindseyBuss\AppData\Local\Temp\ipykernel_2620\3636836726.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  product_data['product_group_num'] = product_data['product_id'].map(cleaned_product_df.set_index('product_id')['product_group'])


   product_id warehouse_section quantity_unit product_group_num
0      000002               SHL            St                31
24     000002               AKL            St                31
25     000003               SHL            St                31
35     000004               SHL            St                31
46     000007               AKL            St                35


In [35]:
def create_product_groups(df):
   
    # Drop product IDs & descriptions
    df = df.drop(['product_id', 'product_description'], axis=1)
    
    # Remove missing values
    df = df.dropna()

    # Remove duplicates
    df = df.drop_duplicates()
    
    # Separate product group number from product group name
    df['product_group_num'] = df['product_group'].apply(lambda x: x.split('_')[0])
    df['product_group_name'] = df['product_group'].apply(lambda x: x.split('_')[1])
    # Remove original product group column
    df = df.drop('product_group', axis=1)
    
    df = df.sort_values(by='product_group_num')

    return df

In [ ]:
product_groups = create_product_groups(product_groups_df)
print(product_groups)
# Export product groups
# product_groups.to_csv()

      product_group_num          product_group_name
1                    16              Sonderverkäufe
383                  18                 Haustechnik
170                  19                    Werkzeug
44857              2000                        Alka
30                   20                   C-Artikel
63                   31  Install.-Befestigungs-Mat.
3                    32                    Schalter
29                   33                Schaltgeräte
15                   34                   Verteiler
0                    35                    Leuchten
79                   36         Entladungs-Leuchten
84                   37                Leuchtmittel
110                  38                   Leitungen
5945                 39                     Metalle
28                   40                    Sprechen
859                  41                     Antenne
1742                 42             Netzwerktechnik
39709                61                    Kataloge


Generate date table.

In [62]:
def create_date_table(df):
    '''Accepts a dataframe containing the order summary information
    and returns a dataframe of the unique dates therein.'''
    # Pull list of unique dates from order summary dataframe.
    df = df['date'].unique()
    # Convert series to datetime
    df = pd.to_datetime(df)
    # Create date table & add year, month, day, and day of week.
    date_table = pd.DataFrame(df, columns=['date'])
    date_table['year'] = date_table['date'].dt.year
    date_table['month'] = date_table['date'].dt.month
    date_table['day'] = date_table['date'].dt.day
    date_table['day_of_week'] = date_table['date'].dt.day_name()

    return date_table



In [ ]:
date_table= create_date_table(final_order_data)
print(date_table.head())
# Export date table.
# date_table.to_csv()

        date  year  month  day day_of_week
0 2017-01-02  2017      1    2      Monday
1 2017-01-05  2017      1    5    Thursday
2 2017-01-03  2017      1    3     Tuesday
3 2017-01-04  2017      1    4   Wednesday
4 2017-01-06  2017      1    6      Friday
